### 子图的流式输出

`chapter05/01-stream&astream.ipynb` 讲了 `stream_mode` 的各种模式。这里补上一个关键问题：**当节点是子图时，那些流式输出该怎么收？**

结论先摆出来：

- 不加 `subgraphs=True`：子图的流被**整个吞掉**，你只能看到父节点那一个 chunk。调 `custom`、`messages` 都会「莫名其妙没输出」。
- 加 `subgraphs=True`：产出变成 `(namespace, chunk)`，子图内部每一步都单独冒出来。`namespace` 是元组，空元组 `()` 表示顶层。

| 调用方式 | 产出形状 |
| --- | --- |
| `stream(...)` | `chunk` |
| `stream(..., subgraphs=True)` | `(namespace, chunk)` |
| `stream_mode=["a","b"]` | `(mode, chunk)` |
| `stream_mode=["a","b"], subgraphs=True` | `(namespace, mode, chunk)` **三元组** |

#### 准备两张图

父图 `top` 有三个节点 `prepare → sub → after`，其中 `sub` 是一张编译好的子图。子图内部只有一个 `work` 节点。

In [1]:
import operator
from typing import Annotated, TypedDict

from langgraph.config import get_stream_writer
from langgraph.graph import END, START, StateGraph


class State(TypedDict):
    log: Annotated[list[str], operator.add]


def make_subgraph():
    """一张两节点的子图，内部会推 custom 进度。"""

    def work(state: State):
        get_stream_writer()({"stage": "work", "pct": 50})
        print("    · 子图的 work 节点执行了")
        return {"log": ["work"]}

    def finish(state: State):
        get_stream_writer()({"stage": "finish", "pct": 90})
        print("    · 子图的 finish 节点执行了")
        return {"log": ["finish"]}

    builder = StateGraph(State)
    builder.add_node("work", work)
    builder.add_node("finish", finish)
    builder.add_edge(START, "work")
    builder.add_edge("work", "finish")
    builder.add_edge("finish", END)
    return builder.compile()


subgraph = make_subgraph()

top = StateGraph(State)
def prepare(state: State):
    get_stream_writer()({"stage": "prepare", "pct": 10})
    return {"log": ["prepare"]}


top.add_node("prepare", prepare)
top.add_node("sub", subgraph)
top.add_node("after", lambda s: {"log": ["after"]})
top.add_edge(START, "prepare")
top.add_edge("prepare", "sub")
top.add_edge("sub", "after")
top.add_edge("after", END)

graph = top.compile()

### 1. 不加 `subgraphs=True`：子图的流被吞掉

`updates` 模式下还能看到子图跑完了（`sub` 那个 chunk 带着子图的完整 state），但**过程完全看不见**——两个 `print` 直接跑到了控制台，流里一个都没有。

In [2]:
print("--- updates，不加 subgraphs ---")
for chunk in graph.stream({"log": []}, stream_mode="updates"):
    print("  ", chunk)

--- updates，不加 subgraphs ---
   {'prepare': {'log': ['prepare']}}
    · 子图的 work 节点执行了
    · 子图的 finish 节点执行了
   {'sub': {'log': ['prepare', 'work', 'finish']}}
   {'after': {'log': ['after']}}


`custom` 模式更能说明问题：**父图的进度能收到，子图的一滴都收不到**。这是最常见的「我明明调了 `get_stream_writer()` 却没输出」的原因。

In [3]:
print("--- custom，不加 subgraphs：只有 prepare 那条 ---")
for payload in graph.stream({"log": []}, stream_mode="custom"):
    print("  ", payload)

--- custom，不加 subgraphs：只有 prepare 那条 ---
   {'stage': 'prepare', 'pct': 10}
    · 子图的 work 节点执行了
    · 子图的 finish 节点执行了


### 2. 加 `subgraphs=True`：子图内部逐步可见

产出变成 `(namespace, chunk)`。namespace 是元组，里面每一段形如 `"节点名:task_id"`，用 `|` 拼接层级——**空元组就是顶层图**。

In [4]:
print("--- updates + subgraphs=True ---")
for namespace, chunk in graph.stream({"log": []}, stream_mode="updates", subgraphs=True):
    print(f"  {str(namespace):<50} {chunk}")

--- updates + subgraphs=True ---
  ()                                                 {'prepare': {'log': ['prepare']}}
    · 子图的 work 节点执行了
    · 子图的 finish 节点执行了
  ('sub:aaab771b-d297-b9ed-1c9a-5e91b0240cc1',)      {'work': {'log': ['work']}}
  ('sub:aaab771b-d297-b9ed-1c9a-5e91b0240cc1',)      {'finish': {'log': ['finish']}}
  ()                                                 {'sub': {'log': ['prepare', 'work', 'finish']}}
  ()                                                 {'after': {'log': ['after']}}


In [5]:
print("\n--- custom + subgraphs=True：子图的进度也出来了 ---")
for namespace, payload in graph.stream({"log": []}, stream_mode="custom", subgraphs=True):
    print(f"  {str(namespace):<50} {payload}")


--- custom + subgraphs=True：子图的进度也出来了 ---
  ()                                                 {'stage': 'prepare', 'pct': 10}
    · 子图的 work 节点执行了
    · 子图的 finish 节点执行了
  ('sub:31f458fa-cdc0-283a-b5b7-38f55be23815',)      {'stage': 'work', 'pct': 50}
  ('sub:31f458fa-cdc0-283a-b5b7-38f55be23815',)      {'stage': 'finish', 'pct': 90}


### 3. 实用技巧：把 namespace 变成可读标签

namespace 里那串 UUID 对人没意义。实际写代码时拼个标签，顺便把层级深度也带出来，方便前端缩进渲染。

In [6]:
def label(namespace: tuple[str, ...]) -> str:
    """('sub:uuid',) -> '顶层 > sub'；多层会拼成 '顶层 > mid > inner'。"""
    if not namespace:
        return "顶层"
    names = [seg.split(":")[0] for seg in namespace]
    return " > ".join(["顶层", *names])


for namespace, chunk in graph.stream({"log": []}, stream_mode="updates", subgraphs=True):
    print(f"  {label(namespace):<24} {chunk}")

  顶层                       {'prepare': {'log': ['prepare']}}
    · 子图的 work 节点执行了
    · 子图的 finish 节点执行了
  顶层 > sub                 {'work': {'log': ['work']}}
  顶层 > sub                 {'finish': {'log': ['finish']}}
  顶层                       {'sub': {'log': ['prepare', 'work', 'finish']}}
  顶层                       {'after': {'log': ['after']}}


### 4. ⚠️ 多模式 + `subgraphs=True` 是**三元组**

这是最容易写错的地方。`stream_mode` 传列表时本来是 `(mode, chunk)`，再叠加 `subgraphs=True` 就变成 `(namespace, mode, chunk)`——**顺序是先 namespace、后 mode**。

In [7]:
# 两元组
for mode, chunk in graph.stream({"log": []}, stream_mode=["custom", "updates"]):
    print("  2-tuple:", mode, str(chunk)[:52])

print()

# 三元组
for namespace, mode, chunk in graph.stream(
    {"log": []}, stream_mode=["custom", "updates"], subgraphs=True
):
    print("  3-tuple:", label(namespace), "|", mode, "|", str(chunk)[:44])

  2-tuple: custom {'stage': 'prepare', 'pct': 10}
  2-tuple: updates {'prepare': {'log': ['prepare']}}
    · 子图的 work 节点执行了
    · 子图的 finish 节点执行了
  2-tuple: updates {'sub': {'log': ['prepare', 'work', 'finish']}}
  2-tuple: updates {'after': {'log': ['after']}}

  3-tuple: 顶层 | custom | {'stage': 'prepare', 'pct': 10}
  3-tuple: 顶层 | updates | {'prepare': {'log': ['prepare']}}
    · 子图的 work 节点执行了
    · 子图的 finish 节点执行了
  3-tuple: 顶层 > sub | custom | {'stage': 'work', 'pct': 50}
  3-tuple: 顶层 > sub | updates | {'work': {'log': ['work']}}
  3-tuple: 顶层 > sub | custom | {'stage': 'finish', 'pct': 90}
  3-tuple: 顶层 > sub | updates | {'finish': {'log': ['finish']}}
  3-tuple: 顶层 | updates | {'sub': {'log': ['prepare', 'work', 'finish'
  3-tuple: 顶层 | updates | {'after': {'log': ['after']}}


如果你想「父图只要最终的，子图要全部过程」，可以按 namespace 过滤：

```python
for namespace, mode, chunk in graph.stream(..., stream_mode=["updates"], subgraphs=True):
    if namespace:            # 只关心子图内部
        ...
    else:                    # 顶层
        ...
```

In [8]:
# 只收顶层 + 只收子图内部，分开处理
top_level, inner = [], []
for namespace, mode, chunk in graph.stream(
    {"log": []}, stream_mode=["custom", "updates"], subgraphs=True
):
    (top_level if not namespace else inner).append((mode, chunk))

print("顶层事件 =", top_level)
print("子图事件 =", inner)

    · 子图的 work 节点执行了
    · 子图的 finish 节点执行了
顶层事件 = [('custom', {'stage': 'prepare', 'pct': 10}), ('updates', {'prepare': {'log': ['prepare']}}), ('updates', {'sub': {'log': ['prepare', 'work', 'finish']}}), ('updates', {'after': {'log': ['after']}})]
子图事件 = [('custom', {'stage': 'work', 'pct': 50}), ('updates', {'work': {'log': ['work']}}), ('custom', {'stage': 'finish', 'pct': 90}), ('updates', {'finish': {'log': ['finish']}})]


### 5. `messages` 模式必须加 `subgraphs=True`

这一条对做打字机效果最关键。**不加 `subgraphs=True` 时，子图里的模型只会给你一条完整消息（`AIMessage`），token 级增量全丢**；加上之后才有 `AIMessageChunk` 流。

顺带注意：`messages` + `subgraphs=True` 的产出是 `(namespace, (chunk, metadata))`——**内层还是二元组**，解包要写两层。

In [9]:
from langchain_core.messages import HumanMessage
from langchain_deepseek import ChatDeepSeek
from langgraph.graph import MessagesState

from dotenv import load_dotenv

load_dotenv(override=True)
llm = ChatDeepSeek(model="deepseek-flash")


def chat(state: MessagesState):
    return {"messages": [llm.invoke([HumanMessage(content="用一句话介绍 LangGraph")])]}


chat_sub = StateGraph(MessagesState)
chat_sub.add_node("chat", chat)
chat_sub.add_edge(START, "chat")
chat_sub.add_edge("chat", END)
chat_subgraph = chat_sub.compile()

chat_top = StateGraph(MessagesState)
chat_top.add_node("sub", chat_subgraph)
chat_top.add_edge(START, "sub")
chat_top.add_edge("sub", END)
chat_graph = chat_top.compile()

In [10]:
# ❌ 不加 subgraphs：只有 1 条完整消息，不是增量
chunks = list(chat_graph.stream({"messages": []}, stream_mode="messages"))
print("不加 subgraphs 的 chunk 数 =", len(chunks))
print("chunk 类型 =", type(chunks[0][0]).__name__)          # AIMessage，不是 AIMessageChunk
print("内容一次性给全 =", chunks[0][0].content[:40], "...")

不加 subgraphs 的 chunk 数 = 1
chunk 类型 = AIMessage
内容一次性给全 = LangGraph 是 LangChain 生态中用于以图结构编排有状态、可循环 ...


In [11]:
# ✅ 加 subgraphs=True：token 级增量回来了
count, shown = 0, 0
for namespace, (chunk, meta) in chat_graph.stream(
    {"messages": []}, stream_mode="messages", subgraphs=True
):
    count += 1
    # DeepSeek 的思考内容走 reasoning_content，正文走 content；开头几个 chunk 可能两者都空
    text = chunk.additional_kwargs.get("reasoning_content") or chunk.content
    if text and shown < 6:
        shown += 1
        print(f"  {type(chunk).__name__:<17} {text[:24]!r}")

print(f"加 subgraphs 的 chunk 数 = {count}（远多于上一节的 1）")

  AIMessageChunk    '我们需要'
  AIMessageChunk    '回答'
  AIMessageChunk    '用户'
  AIMessageChunk    '中文'
  AIMessageChunk    '请求'
  AIMessageChunk    '：“'


加 subgraphs 的 chunk 数 = 161（远多于上一节的 1）


两个细节：

- `metadata["langgraph_node"]` 是**子图内部节点名**（`chat`），不是父节点名（`sub`）。
- `metadata["langgraph_checkpoint_ns"]` 是**用 `|` 拼起来的完整层级**，形如 `sub:<id>|chat:<id>`——和第 6 节的 namespace 规则一致，只是这里多了模型所在节点那一段。

In [12]:
nodes, namespaces = set(), set()
for namespace, (chunk, meta) in chat_graph.stream(
    {"messages": []}, stream_mode="messages", subgraphs=True
):
    nodes.add(meta.get("langgraph_node"))
    namespaces.add(meta.get("langgraph_checkpoint_ns"))

print("langgraph_node =", nodes)
print("langgraph_checkpoint_ns =", namespaces)

langgraph_node = {'chat'}
langgraph_checkpoint_ns = {'sub:df1fb4ec-3d0b-4d72-8399-163cae0d8df0|chat:1bfed419-6911-450b-f330-3daeaf1636d1'}


### 6. 三层嵌套：namespace 会越拼越长

递归子图（子图里还有子图）时，每一层都往 namespace 里追加一段 `节点名:task_id`，用 `|` 连接。这是定位「到底是哪一层出的问题」最直接的办法。

In [13]:
# 最内层
inner_builder = StateGraph(State)
inner_builder.add_node("i1", lambda s: {"log": ["i1"]})
inner_builder.add_edge(START, "i1")
inner_builder.add_edge("i1", END)
inner_graph = inner_builder.compile()

# 中间层：内含 inner
mid_builder = StateGraph(State)
mid_builder.add_node("m1", lambda s: {"log": ["m1"]})
mid_builder.add_node("inner", inner_graph)
mid_builder.add_edge(START, "m1")
mid_builder.add_edge("m1", "inner")
mid_builder.add_edge("inner", END)
mid_graph = mid_builder.compile()

# 顶层：内含 mid
top2 = StateGraph(State)
top2.add_node("t1", lambda s: {"log": ["t1"]})
top2.add_node("mid", mid_graph)
top2.add_edge(START, "t1")
top2.add_edge("t1", "mid")
top2.add_edge("mid", END)
nested = top2.compile()

for namespace, chunk in nested.stream({"log": []}, stream_mode="updates", subgraphs=True):
    print(f"  {label(namespace):<28} {chunk}")

  顶层                           {'t1': {'log': ['t1']}}
  顶层 > mid                     {'m1': {'log': ['m1']}}
  顶层 > mid > inner             {'i1': {'log': ['i1']}}
  顶层 > mid                     {'inner': {'log': ['t1', 'm1', 'i1']}}
  顶层                           {'mid': {'log': ['t1', 'm1', 't1', 'm1', 'i1']}}


### 7. 并行子图：靠 task id 区分

父图里挂两个子图节点并行跑时，两路的 namespace **前缀不同**（`s1:...` 和 `s2:...`），可以据此分流。想在 UI 上分别渲染进度条，就靠这个前缀。

In [14]:
def make(name):
    b = StateGraph(State)
    b.add_node(name, lambda s: {"log": [name]})
    b.add_edge(START, name)
    b.add_edge(name, END)
    return b.compile()


parallel = StateGraph(State)
parallel.add_node("s1", make("sub1"))
parallel.add_node("s2", make("sub2"))
parallel.add_edge(START, "s1")
parallel.add_edge(START, "s2")
parallel.add_edge("s1", END)
parallel.add_edge("s2", END)
parallel_graph = parallel.compile()

for namespace, chunk in parallel_graph.stream({"log": []}, stream_mode="updates", subgraphs=True):
    print(f"  {label(namespace):<28} {chunk}")

  顶层 > s2                      {'sub2': {'log': ['sub2']}}
  顶层 > s1                      {'sub1': {'log': ['sub1']}}
  顶层                           {'s1': {'log': ['sub1']}}
  顶层                           {'s2': {'log': ['sub2']}}


### 8. `tasks` 模式：子图的 task 也带 namespace

`tasks` 模式本来就是看「每个 task 的开始与结束」，配上 `subgraphs=True` 后连子图内部的 task 也能逐个列出——父节点自己的那一条带 `input`/`result`，子图的同理。

In [15]:
for namespace, task in parallel_graph.stream({"log": []}, stream_mode="tasks", subgraphs=True):
    phase = "开始" if "input" in task else "结束"
    print(f"  {label(namespace):<22} {phase}  task_id={task['id'][:8]}  name={task['name']}")

  顶层                     开始  task_id=815f91c4  name=s1
  顶层                     开始  task_id=8a0338c0  name=s2
  顶层 > s2                开始  task_id=a43eecaf  name=sub2
  顶层 > s2                结束  task_id=a43eecaf  name=sub2
  顶层 > s1                开始  task_id=570a48d7  name=sub1
  顶层 > s1                结束  task_id=570a48d7  name=sub1
  顶层                     结束  task_id=815f91c4  name=s1
  顶层                     结束  task_id=8a0338c0  name=s2


### 9. `astream` 完全一样

异步版参数与产出形状和同步版一一对应，改 `async for` 即可。做 Web 服务（SSE / WebSocket）时用这个。

In [16]:
import asyncio


async def main():
    print("--- astream + subgraphs=True ---")
    async for namespace, chunk in graph.astream(
        {"log": []}, stream_mode="updates", subgraphs=True
    ):
        print(f"  {label(namespace):<28} {chunk}")

    print("\n--- astream + 多模式 + subgraphs=True（三元组）---")
    async for namespace, mode, chunk in graph.astream(
        {"log": []}, stream_mode=["custom", "updates"], subgraphs=True
    ):
        print(f"  {label(namespace):<22} | {mode:<8} | {str(chunk)[:40]}")


await main()

--- astream + subgraphs=True ---
  顶层                           {'prepare': {'log': ['prepare']}}
    · 子图的 work 节点执行了
    · 子图的 finish 节点执行了
  顶层 > sub                     {'work': {'log': ['work']}}
  顶层 > sub                     {'finish': {'log': ['finish']}}
  顶层                           {'sub': {'log': ['prepare', 'work', 'finish']}}
  顶层                           {'after': {'log': ['after']}}

--- astream + 多模式 + subgraphs=True（三元组）---
  顶层                     | custom   | {'stage': 'prepare', 'pct': 10}
  顶层                     | updates  | {'prepare': {'log': ['prepare']}}
    · 子图的 work 节点执行了
  顶层 > sub               | custom   | {'stage': 'work', 'pct': 50}
    · 子图的 finish 节点执行了
  顶层 > sub               | updates  | {'work': {'log': ['work']}}
  顶层 > sub               | custom   | {'stage': 'finish', 'pct': 90}
  顶层 > sub               | updates  | {'finish': {'log': ['finish']}}
  顶层                     | updates  | {'sub': {'log': ['prepare', 'work', 'fin
  顶层           

### 小结

| 想要的效果 | 写法 |
| --- | --- |
| 看子图内部每一步 | `stream(..., stream_mode="updates", subgraphs=True)`，解包 `(namespace, chunk)` |
| 看子图内部 `custom` 进度 | 同上，换 `stream_mode="custom"` |
| 同时要多种视图 | `stream_mode=["updates", "custom"], subgraphs=True`，解包**三元组** `(namespace, mode, chunk)` |
| 子图里的打字机效果 | `stream_mode="messages", subgraphs=True`，解包**两层** `(namespace, (chunk, meta))` |
| 定位第几层出的问题 | `namespace` 每段形如 `节点名:task_id`，多层用 `\|` 拼接 |
| 并行子图分别渲染 | 按 namespace 的首段（父节点名）分流 |

**最该记住的一条**：`get_stream_writer()` 在子图里写了却收不到，99% 是忘了 `subgraphs=True`。